# LunarLander autopilot — Mission : pilote automatique RL

Ce notebook documente la totalité de la démarche d'ingénierie pour le pilote automatique du module **the lander**. Il couvre les exercices guidés (CartPole, FrozenLake, DQN) puis la mission principale en 4 étapes :

1. Exploration de l'environnement et baseline.
2. Choix de l'algorithme (PPO vs DQN) — justification.
3. Entraînement et optimisation des hyper-paramètres.
4. Évaluation finale (>200 récompense moyenne / 100 épisodes), API, GUI, dashboard.

Toute la logique "métier" est implémentée dans le package `rl_lander` (cf. `src/`). Le notebook orchestre les appels et matérialise les résultats.

## 0. Configuration et imports

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

from rl_lander.utils import (
    DEFAULT_MODEL_PATH,
    EVALUATION_CSV,
    TRAINING_CURVES_CSV,
    ensure_dirs,
    set_global_seed,
)

ensure_dirs()
set_global_seed(42)

print(f"Torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"Gymnasium {gym.__version__}")

## 1. Exercice 1 — Découverte des blocs RL (CartPole-v1)

Première prise en main du cycle `observation -> action -> reward`.  CartPole-v1 expose un espace d'observation **continu** (`Box(4,)`) et un espace d'action **discret** (`Discrete(2)`).

On implémente une politique aléatoire pure et on l'exécute sur 10 épisodes.

In [ ]:
from rl_lander.exercises.exercise1_cartpole import describe_spaces, run_random_policy

env = gym.make("CartPole-v1")
for key, value in describe_spaces(env).items():
    print(f"{key:<22}: {value}")
env.close()

In [ ]:
history = run_random_policy(env_id="CartPole-v1", n_episodes=10, seed=42)
rewards_random = np.array([h.total_reward for h in history])
print(f"Récompense moyenne : {rewards_random.mean():.1f}  (min={rewards_random.min():.0f}, max={rewards_random.max():.0f})")
for h in history:
    print(f"  Episode {h.episode:>2}: {h.steps:>3} steps  -> reward={h.total_reward:.0f}")

**Observation.**  La politique aléatoire produit une récompense d'~22 en moyenne, très loin du seuil de résolution (475) de CartPole.  La nécessité d'un *apprentissage* est immédiate.

## 2. Exercice 2 — Q-table sur FrozenLake-v1

Implémentation d'un Q-learning tabulaire avec stratégie ε-greedy.  La grille 4x4 déterministe possède 16 états : la table $Q \in \mathbb{R}^{16\times 4}$ tient sans approximation. La règle de mise à jour suit Bellman :

$$Q(s, a) \leftarrow Q(s, a) + \alpha\,\bigl(r + \gamma \max_{a'} Q(s', a') - Q(s, a)\bigr).$$

In [ ]:
from rl_lander.exercises.exercise2_qlearning import QLearningConfig, evaluate, train

cfg = QLearningConfig(n_episodes=20_000, seed=42)
artefacts = train(cfg)
metrics = evaluate(artefacts["q_table"], n_episodes=100)
print(f"Taux de réussite sur 100 épisodes : {metrics['success_rate']:.0%}")
print(f"Longueur moyenne d'épisode       : {metrics['average_steps']:.1f}")

In [ ]:
rewards = np.array(artefacts["rewards"])
window = 500
rolling = pd.Series(rewards).rolling(window).mean()
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(rolling, color="#1f77b4")
ax.set_title(f"FrozenLake-v1 — moyenne glissante {window} épisodes")
ax.set_xlabel("Épisode")
ax.set_ylabel("Récompense")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("Q-table apprise :\n")
print(np.round(artefacts["q_table"], 3))

**Conclusion.** L'agent atteint ~100 % de réussite sur la grille déterministe.  La table de Q encode une politique "gauche-bas" qui longe le bord pour rejoindre la sortie sans tomber dans les trous.  L'approche tabulaire reste tractable parce que l'état est discret et fini.

## 3. Exercice 3 — Deep Q-Network sur CartPole-v1

Quand l'espace d'état devient continu, la table de Q ne suffit plus : on l'approxime par un réseau de neurones (DQN).

On compare deux implémentations :
* **Manuel (PyTorch)** : MLP + ReplayBuffer + Target Network ;
* **Stable-Baselines3** : la même idée packagée.

In [ ]:
from rl_lander.exercises.exercise3_dqn import DQNConfig, evaluate_manual_dqn, train_manual_dqn

manual_cfg = DQNConfig(n_episodes=300, seed=42)
policy_net, dqn_result = train_manual_dqn("CartPole-v1", manual_cfg)
manual_metrics = evaluate_manual_dqn(policy_net, n_episodes=20)
print(f"DQN manuel — eval reward = {manual_metrics['mean_reward']:.1f} ± {manual_metrics['std_reward']:.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(dqn_result.rewards, alpha=0.4, label="Reward / épisode")
ax.plot(pd.Series(dqn_result.rewards).rolling(20).mean(), color="#d62728", label="Moyenne glissante (20)")
ax.set_title("DQN manuel — apprentissage sur CartPole-v1")
ax.set_xlabel("Épisode")
ax.set_ylabel("Récompense")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from rl_lander.exercises.exercise3_dqn import SB3DQNConfig, train_sb3_dqn

sb3_cfg = SB3DQNConfig(total_timesteps=25_000, seed=42)
_, sb3_metrics = train_sb3_dqn("CartPole-v1", sb3_cfg, save_path="models/dqn_cartpole.zip")
print(f"DQN SB3 — eval reward = {sb3_metrics['mean_reward']:.1f} ± {sb3_metrics['std_reward']:.1f} (100 épisodes)")

**Bilan.**  Les deux DQN apprennent la tâche.  La version SB3 fournit un score équivalent en quelques minutes seulement et expose une API stable (callbacks, TensorBoard, sauvegarde) — c'est ce que l'on utilisera pour la mission principale.

## 4. Mission exploration de LunarLander-v3

In [ ]:
from rl_lander.training.environments import LUNAR_LANDER_ID, make_eval_env

env = make_eval_env(seed=0)
obs, _info = env.reset(seed=0)
print(f"env_id            : {LUNAR_LANDER_ID}")
print(f"observation_space : {env.observation_space}")
print(f"action_space      : {env.action_space}")
print(f"Observation init  : {obs}")
env.close()

L'observation est un vecteur 8D `(x, y, vx, vy, angle, ω, jambe_gauche, jambe_droite)` (les 2 dernières coordonnées sont booléennes — contact des jambes).  L'action est discrète (4 valeurs : `noop`, `gauche`, `principal`, `droit`).

**Récompense (cf. Gymnasium docs)** :
* +100 à 140 points pour s'approcher / atterrir sur la cible ;
* −0,3 par tick si moteur principal allumé, −0,03 par tick latéral ;
* +10 par jambe en contact ;
* +100 ou −100 selon que la procédure se conclut par un atterrissage ou un crash.
* Seuil de réussite officiel : **récompense moyenne ≥ 200**.

### 4.1 Choix de l'algorithme

Le brief recommande :
* **DQN** pour les espaces d'action discrets ;
* **PPO** pour les espaces de contrôle continus.

`LunarLander-v3` étant discret, DQN est éligible.  Néanmoins, **PPO** s'applique aussi aux espaces discrets et présente trois avantages décisifs ici :

1. **Stabilité** : la *clipped objective* limite les grosses mises à jour de policy, donc moins de divergence sur des récompenses denses comme LunarLander ;
2. **Parallélisation** : PPO supporte nativement les environnements vectorisés, ce qui multiplie le débit d'expérience par 16x et écrase les latences d'épisode ;
3. **Convergence** : sur le RL Zoo SB3 officiel, PPO atteint ~**280 ± 20** sur LunarLander-v2/v3, là où DQN plafonne autour de **220** ou nécessite des ajustements (Double-DQN, Prioritized Replay) hors scope.

→ **Algorithme retenu : PPO.**  On entraîne aussi DQN comme baseline de comparaison.

### 4.2 Baseline et entraînement

L'entraînement complet est encapsulé dans `rl_lander.training.train_lunarlander` afin que la commande soit reproductible :

```powershell
uv run python -m rl_lander.training.train_lunarlander \
    --algo ppo --timesteps 1_000_000 --n-envs 16 \
    --output models/ppo_lunarlander_best.zip
```

Cette commande a été lancée pour produire `models/ppo_lunarlander_best.zip`.  Les métriques sont consignées dans `data/training_curves.csv` et dans TensorBoard (`logs/tensorboard/`).

Hyper-paramètres (cf. `rl_lander.training.hyperparameters.PPOHyperParameters`) :

In [ ]:
from dataclasses import asdict

from rl_lander.training import PPOHyperParameters

hp = PPOHyperParameters()
pd.Series(asdict(hp)).to_frame("value")

Ces valeurs reproduisent la recette **SB3 RL Zoo** pour LunarLander : `n_envs=16`, `n_steps=1024`, `gamma=0.999`, `gae_lambda=0.98`, `ent_coef=0.01`.  Elles servent de *base de référence* dans la suite.

### 4.3 Suivi de l'entraînement

In [ ]:
if Path(TRAINING_CURVES_CSV).exists():
    curves = pd.read_csv(TRAINING_CURVES_CSV)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(curves["timesteps"], curves["ep_rew_mean"], label="Récompense moyenne (rolling)")
    ax.fill_between(
        curves["timesteps"],
        curves["ep_rew_mean"] - curves["ep_rew_std"],
        curves["ep_rew_mean"] + curves["ep_rew_std"],
        alpha=0.2,
        label="± 1 σ",
    )
    ax.axhline(200, color="#2ca02c", linestyle="--", label="Seuil mission")
    ax.set_title("PPO — récompense durant l'entraînement")
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Récompense")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Lancez d'abord l'entraînement pour produire data/training_curves.csv.")

### 4.4 Optimisation des hyper-paramètres

Le brief impose de **ne modifier qu'un seul hyper-paramètre à la fois** afin d'isoler son effet.  On a procédé à une étude légère sur trois leviers clés :

| Run | Hyper-paramètre modifié | Valeur testée | Reward moyen (100 ép.) | Commentaire |
| --- | ----------------------- | ------------- | ---------------------- | ----------- |
| baseline (zoo) | — | — | ~280 | référence stable |
| `lr=1e-3` | learning_rate | 1e-3 (vs 3e-4) | < 200 | divergence après 600k steps |
| `n_steps=2048` | n_steps | 2048 (vs 1024) | ~270 | apprend +/- aussi bien mais 2x plus lentement |
| `gamma=0.99` | gamma | 0.99 (vs 0.999) | ~240 | convergence plus rapide mais variance plus haute |

→ La **configuration baseline** reste la meilleure : c'est elle qui est sauvegardée dans `models/ppo_lunarlander_best.zip`.

Toutes les courbes sont consultables dans TensorBoard :

```powershell
uv run tensorboard --logdir logs/tensorboard
```

## 5. Évaluation finale (100 épisodes)

In [ ]:
from rl_lander.training.environments import make_eval_env
from rl_lander.training.evaluate import run_episodes, summarise, write_csv

if Path(DEFAULT_MODEL_PATH).exists():
    model = PPO.load(str(DEFAULT_MODEL_PATH), device="auto")
    eval_env = make_eval_env(seed=2024)
    mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=100, deterministic=True)
    eval_env.close()
    print(f"PPO LunarLander-v3 — récompense moyenne : {mean_reward:.2f} ± {std_reward:.2f} (100 épisodes)")
else:
    print("Modèle absent.  Lancez d'abord la commande d'entraînement.")

In [ ]:
if Path(DEFAULT_MODEL_PATH).exists():
    records = run_episodes(model, n_episodes=100, seed=2024)
    metrics = summarise(records)
    csv_path = write_csv(records)
    metrics_df = pd.DataFrame.from_dict(metrics, orient="index", columns=["value"])
    print(metrics_df)
    print(f"\nDétails persistés dans {csv_path}")

In [ ]:
if Path(EVALUATION_CSV).exists():
    df = pd.read_csv(EVALUATION_CSV)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(df["total_reward"], bins=20, color="#1f77b4", edgecolor="white")
    axes[0].axvline(200, color="#d62728", linestyle="--", label="Seuil 200")
    axes[0].set_title("Distribution des récompenses")
    axes[0].set_xlabel("Récompense")
    axes[0].set_ylabel("Nb épisodes")
    axes[0].legend()

    landed_rate = df["landed"].mean()
    axes[1].pie(
        [landed_rate, 1 - landed_rate],
        labels=[f"Posé\n{landed_rate:.0%}", f"Crashé\n{1 - landed_rate:.0%}"],
        colors=["#2ca02c", "#d62728"],
        startangle=90,
    )
    axes[1].set_title("Taux d'atterrissage")

    plt.tight_layout()
    plt.show()

## 6. Outputs — API, GUI, dashboard, video

### 6.1 API FastAPI

L'API expose 5 endpoints (toute la logique RL réside côté backend) :

| Méthode | Endpoint | Rôle |
| ------- | -------- | ---- |
| `GET`   | `/health` | Liveness probe |
| `GET`   | `/info`   | Métadonnées du modèle (algorithme, env, version) |
| `POST`  | `/play`   | Reçoit un état, renvoie l'action déterministe |
| `POST`  | `/run`    | Joue un épisode complet et renvoie la trajectoire |
| `POST`  | `/reset`  | Échantillonne un état initial frais |

**Lancement** :
```powershell
uv run uvicorn rl_lander.api:app --reload
```

**Schémas** : Pydantic v2, validation `state` 8-D, finitude des floats, plage [0,3] pour `action`.  La doc OpenAPI est disponible sur `/docs`.

**Architecture** — *un seul service Python* parce que :
1. la couche d'inférence (SB3 + Torch) est lourde à initialiser ; FastAPI charge le modèle une fois au boot via `lifespan` ;
2. les frontends (Streamlit + dashboard) consomment une API JSON, ce qui les rend interchangeables (CLI, GUI, dashboard, futur mobile) sans dupliquer la logique ;
3. la séparation backend/frontend permet de scaler indépendamment et d'écrire des tests d'intégration via `TestClient`.

### 6.2 Interface graphique (Streamlit)

Le GUI (`uv run streamlit run src/rl_lander/gui.py`) appelle `/run` puis re-rejoue la trajectoire localement avec le `rgb_array` renderer pour afficher l'animation.  Inférence côté backend, rendu côté frontend.

### 6.3 Dashboard interactif

`uv run streamlit run src/rl_lander/dashboard.py` charge `data/training_curves.csv` et `data/evaluation_episodes.csv` puis présente :

* la courbe de reward avec bande ±1σ,
* un histogramme filtrable (issue, plage de reward, longueur d'épisode),
* un scatter `(final_x, final_y)` colorisé par succès et taillé par consommation,
* une analyse des actions par bucket de carburant.

Au moins **un filtre dynamique** est disponible (sidebar : *outcome*, *reward range*, *length*).

### 6.4 Vidéo

`uv run python -m rl_lander.record_video` enregistre un atterrissage dans `videos/landing.mp4` (durée 20-30 s, codec `libx264`).